# 4. Run an experiment for the installed method

Experiments can run for hours, so the runner is started in the background on `control` (it keeps running if
this notebook or the Jupyter session is closed). Use the status cell to follow progress.

Every run is stored on `control` in `~/experiment-results/<method>-<timestamp>/` with its `runner.log`.
The runner's experiment subfolder contains `run-configuration.json` (settings, branch and commit, images,
node sites/hosts and pod placement), alongside `experiment.json` and the `run-*` folders. The results are kept
when DYNAMOS is reset or reinstalled, so they stay available over SSH. The download cell additionally copies a
run into `results/` here as a zip file.

In [ ]:
METHOD = "semi-overlap-fed-bcd"   # must match the method installed by 03_install_method  # "baseline" | "fed-bcd" | "semi-overlap-fed-bcd" | "fed-encrypt-dynamic"
CYCLES = None         # None = runner default (120 local steps; 12 rounds for fed-encrypt-dynamic)
RUNS = 1
EXTRA_ARGS = ""       # e.g. "--communication-frequency 10 --sample-batch-size 64" or "--policy-removal 4 --policy-reintroduction 8"

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

import deployment_config as cfg
import fabric_helpers as fh
from fabric_helpers import run, run_script

fablib = fablib_manager()
slice_ = fh.get_slice(fablib)
nodes = fh.nodes_by_name(slice_)
control = nodes[cfg.CONTROL_NODE]
dynamos = nodes[cfg.DYNAMOS_NODE]
print(f"Slice {slice_.get_name()}: {slice_.get_state()}, lease end {slice_.get_lease_end()}")

In [ ]:
import datetime
import shlex

assert METHOD in cfg.METHODS, f"Unknown method {METHOD!r}"
installed = run(control, f"cat {cfg.INSTALLED_METHOD_FILE} 2>/dev/null || true", quiet=True).strip()
assert installed == METHOD, f"Installed method is {installed or 'none'!r}, not {METHOD!r}; run 03_install_method first"

checkout = cfg.checkout_dir(METHOD)
python = f"$HOME/{cfg.EXPERIMENT_VENV}/bin/python"
run(control, f"{python} -m pip install -q -r $HOME/{checkout}/{cfg.METHODS[METHOD]['requirements']}", quiet=True)
print(run(control, f"{python} $HOME/{checkout}/{cfg.METHODS[METHOD]['runner']} --help", quiet=True))

## Start the experiment in the background

In [ ]:
running = run(control, f"test -f {cfg.CURRENT_RUN_FILE} && kill -0 $(cut -d' ' -f2 {cfg.CURRENT_RUN_FILE}) 2>/dev/null && echo yes || echo no", quiet=True).strip()
assert running == "no", "An experiment is still running; check or stop it below first"

run_id = f"{METHOD}-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"
output_dir = f"$HOME/{cfg.RESULTS_DIR}/{run_id}"
log_file = f"{output_dir}/runner.log"
args = f"--runs {RUNS} --output-dir {output_dir}"
if CYCLES:
    args += f" --cycles {CYCLES}"
if EXTRA_ARGS:
    args += f" {EXTRA_ARGS}"

settings = {"run_id": run_id, "method": METHOD, "cycles": CYCLES, "runs": RUNS, "extra_args": EXTRA_ARGS,
            "command": f"{cfg.METHODS[METHOD]['runner']} {args}"}
run(control, f"mkdir -p $HOME/{cfg.RESULTS_DIR} && mkdir {output_dir}", quiet=True)
fh.save_run_configuration(control, output_dir, fh.run_configuration(slice_, control, checkout, settings))
launcher = fh.upload_script(control, "node_scripts/start_experiment.py")

start = (
    f"cd $HOME/{checkout} || exit 1; "
    f"nohup setsid {python} $HOME/{launcher} {output_dir} {python} {cfg.METHODS[METHOD]['runner']} {args} > {log_file} 2>&1 < /dev/null & "
    f"echo \"{run_id} $!\" > $HOME/{cfg.CURRENT_RUN_FILE}; sleep 5; kill -0 $! || {{ tail -n 30 {log_file}; exit 1; }}"
)
run(control, start, quiet=True)
print(f"Started {run_id}: {cfg.METHODS[METHOD]['runner']} {args}")

## Status (rerun whenever you like)

In [ ]:
print(run(control, f"""
read run_id pid < {cfg.CURRENT_RUN_FILE}
if kill -0 "$pid" 2>/dev/null; then echo "$run_id: RUNNING (pid $pid)"; else echo "$run_id: FINISHED"; fi
log=$HOME/{cfg.RESULTS_DIR}/$run_id/runner.log; [ -f "$log" ] || log=$HOME/{cfg.LOGS_DIR}/$run_id.log
echo; tail -n 30 "$log"
""", quiet=True))

### Live results (optional)

Polls the latest `result.json` of the run every `POLL_SECONDS` and redraws two plots until the runner stops
(or you interrupt the cell; the experiment keeps running). The runner refreshes `result.json` about every 5 s.

1. **Accuracy per round.** With the updated `semi-overlap-fed-bcd` API gateway, a round's accuracy appears one round
   after it finishes; on the other methods accuracies may only appear when the run is done.
2. **Share of wall time (cumulative).** Each round's elapsed time on the API gateway's clock is split along the
   critical path: in every phase, the request that replied last determines when the phase ends.
   - *Compute*: that request's algorithm time, as reported by the participant's sidecar.
   - *Communication*: the rest of that request's round trip (network, agent and sidecar handling).
   - *Synchronization & coordination*: the remaining round time outside these requests: gateway work between phases,
     policy checks and, on `semi-overlap-fed-bcd`, waiting for the previous round to free the next pipeline stage.
   - *Not split*: critical requests without sidecar timing (e.g. failed or retried calls).

   Each point is the running total up to that round. Rounds run back to back on `baseline`, `fed-bcd` and
   `fed-encrypt-dynamic`, so this is the share of the training wall time. On `semi-overlap-fed-bcd` rounds overlap,
   so it is the share of the rounds' own elapsed time (latency), not of the run's wall clock.

In [ ]:
import json
import shlex
import time

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import clear_output

RUN_ID = None        # None = the current (or last started) run
POLL_SECONDS = 3
SHOW_ROUNDS = 10

run_id_ = RUN_ID or run(control, f"cut -d' ' -f1 {cfg.CURRENT_RUN_FILE}", quiet=True).strip()
poll = f"""
read current pid < {cfg.CURRENT_RUN_FILE}
if [ "$current" = {shlex.quote(run_id_)} ] && kill -0 "$pid" 2>/dev/null; then echo running; else echo stopped; fi
file=$(ls -t $HOME/{cfg.RESULTS_DIR}/{run_id_}/*/run-*/result.json 2>/dev/null | head -n 1)
echo "$file"; [ -z "$file" ] || cat "$file"
"""
PARTS = (("Compute", "#4477AA"), ("Communication", "#EE7733"),
         ("Synchronization & coordination", "#CC6677"), ("Not split (no compute timing)", "#BBBBBB"))


def fetch():
    state, path, text = (run(control, poll, check=False, quiet=True).split("\n", 2) + ["", ""])[:3]
    try:
        result = json.loads(text) if path.strip() else None
    except json.JSONDecodeError:
        result = None   # caught while the runner was rewriting it; the next poll gets it
    return state.strip() == "running", path.strip(), result


def ms(timing, *keys):
    for key in keys:
        value = timing.get(key)
        if isinstance(value, (int, float)) and not isinstance(value, bool):
            return float(value)
    return None


def wall_split(timing):
    """Split a round's elapsed time into PARTS along the last-replying request of each phase."""
    wall = ms(timing, "round_ms", "cycle_latency_ms", "duration_ms")
    phases = [p for p in (timing.get("phases") or {}).values() if isinstance(p, dict)]
    if wall is None or not phases:
        return None
    compute = communication = unsplit = 0.0
    for phase in phases:
        calls = phase.get("calls") or {}
        calls = [c for c in (calls.values() if isinstance(calls, dict) else calls)
                 if isinstance(c, dict) and ms(c, "duration_ms") is not None]
        if not calls:
            continue
        # fed-encrypt-dynamic lists calls with end times; the other gateways merge them per participant.
        last = max(calls, key=lambda c: ms(c, "end_ms", "duration_ms"))
        duration, algorithm = ms(last, "duration_ms"), ms(last, "algorithm_ms")
        if algorithm is None or last.get("failed") or ms(last, "unmeasured_calls"):
            unsplit += duration
        else:
            compute += algorithm
            communication += duration - algorithm
    return compute, communication, max(wall - compute - communication - unsplit, 0.0), unsplit


def draw(path, result, running):
    rounds = sorted(result.get("results") or [], key=lambda r: r.get("train_round", 0))
    accuracy = [(r["train_round"], r["accuracy"]) for r in rounds if r.get("accuracy") is not None]
    split = [(r["train_round"], parts) for r in rounds if (parts := wall_split(r.get("timing") or {}))]

    figure, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))
    figure.suptitle(f"{run_id_}: {result.get('status')}{' (runner active)' if running else ''}, "
                    f"{len(rounds)} rounds completed")
    if accuracy:
        left.plot(*zip(*accuracy), marker="o", markersize=3)
    left.set(title="Accuracy per round", xlabel="Round", ylabel="Accuracy (%)")
    left.grid(alpha=0.3)

    title = "Share of round wall time (cumulative)"
    if split:
        totals, shares = [0.0] * len(PARTS), [[] for _ in PARTS]
        for _, parts in split:
            totals = [total + part for total, part in zip(totals, parts)]
            for share, total in zip(shares, totals):
                share.append(100 * total / (sum(totals) or 1))
        shown = [i for i, total in enumerate(totals) if total > 0]
        right.stackplot([cycle for cycle, _ in split], *(shares[i] for i in shown),
                        colors=[PARTS[i][1] for i in shown],
                        labels=[f"{PARTS[i][0]} ({shares[i][-1]:.1f} %)" for i in shown])
        right.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
        title += f"\ncommunication + synchronization: {shares[1][-1] + shares[2][-1]:.1f} %"
    right.set(title=title, xlabel="Round", ylabel="% of wall time", ylim=(0, 100))
    for axis in (left, right):
        axis.xaxis.set_major_locator(MaxNLocator(integer=True))
    figure.tight_layout()
    plt.show()

    print(path)
    for r in rounds[-SHOW_ROUNDS:]:
        value = r.get("accuracy")
        latency = ms(r.get("timing") or {}, "cycle_latency_ms", "round_ms", "duration_ms")
        print(f"  round {r.get('train_round'):>5}   accuracy {'-' if value is None else f'{value:6.2f} %':>8}   "
              f"round time {'-' if latency is None else f'{latency / 1000:.2f} s'}")


latest = None
try:
    while True:
        try:
            running, path, result = fetch()
        except Exception as error:   # e.g. a dropped SSH connection; try again on the next poll
            print(f"Poll failed: {error}")
            time.sleep(POLL_SECONDS)
            continue
        latest = (path, result) if result is not None else latest
        clear_output(wait=True)
        if latest:
            draw(*latest, running)
        else:
            print(f"No result.json yet for {run_id_}; the request may still be starting")
        if not running:
            print("The runner is not running; stopped polling.")
            break
        time.sleep(POLL_SECONDS)
except KeyboardInterrupt:
    print("Stopped polling; the experiment keeps running.")

## Stop the running experiment (only if needed)

In [ ]:
STOP = False

if STOP:
    print(run(control, f"read run_id pid < {cfg.CURRENT_RUN_FILE}; kill -TERM -- -$pid && echo stopped $run_id", check=False, quiet=True))

## Results stored on `control`

Lists every run kept on `control` and prints the SSH command to reach them.

In [ ]:
print(run(control, f"cd $HOME/{cfg.RESULTS_DIR} 2>/dev/null && du -sh -- */ || echo 'No results yet'", check=False, quiet=True))
print(f"\nOn control: ~/{cfg.RESULTS_DIR}/<run_id>/")
print(f"SSH: {control.get_ssh_command()}")

## Download a run into `results/` (zip, opens in Windows)

In [ ]:
import os
import zipfile

RUN_ID = None   # None = the most recent run; or a folder name listed above, e.g. "baseline-20261110-141453"

run_id = RUN_ID or run(control, f"cut -d' ' -f1 {cfg.CURRENT_RUN_FILE}", quiet=True).strip()
remote_dir = f"{cfg.RESULTS_DIR}/{run_id}"
legacy_log = f"{cfg.LOGS_DIR}/{run_id}.log"
archive = f"{cfg.REMOTE_TOOLS_DIR}/{run_id}.zip"
# Runs started before the log moved into the run folder keep it in ~/experiment-logs; copy it in.
run(control, f"cd $HOME && test -d {remote_dir} "
             f"&& {{ [ -f {remote_dir}/runner.log ] || [ ! -f {legacy_log} ] || cp {legacy_log} {remote_dir}/runner.log; }} "
             f"&& rm -f {archive} && cd {cfg.RESULTS_DIR} && python3 -m zipfile -c $HOME/{archive} {run_id}", quiet=True)

os.makedirs("results", exist_ok=True)
local_zip = os.path.join("results", f"{run_id}.zip")
control.download_file(local_file_path=local_zip, remote_file_path=archive)
with zipfile.ZipFile(local_zip) as archive_file:
    archive_file.extractall("results")
print(f"Saved to results/{run_id}/ and results/{run_id}.zip")